# Descriptive analytics on Indian prison statistics

## Phase 4 scope

We analyze the 27 approved source views, the 447 state-year population reference and the 437-profile cohort. Each table states its observation unit and coverage. This notebook generates statistical tables and interpretations only; visualizations and mining algorithms remain later work.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src/analytics/descriptive.py").is_file()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
from IPython.display import display
from src.analytics.descriptive import run_descriptive
from src.preprocessing.cleaning import sha256
pd.set_option("display.max_rows", 80)
REPORTS = ROOT / "results/tables/descriptive"

In [2]:
# regenerate the statistical tables from verified inputs
summary = run_descriptive(ROOT)
print(json.dumps(summary, indent=2))

{
  "descriptive_sources": 27,
  "reference_state_years": 447,
  "balanced_states": 34,
  "balanced_state_years": 442,
  "profile_state_years": 437,
  "common_demographic_state_years": 426,
  "numeric_summary_rows": 131,
  "stratified_summary_rows": 2659,
  "tables_generated": 22,
  "recidivism_invalid_count_pairs": 1,
  "upstream_files_verified": 85,
  "upstream_unchanged": true,
  "visualization_or_mining_started": false
}


## Statistical conventions

Variance and standard deviation use ddof=1. Quartiles use linear interpolation and the median is Q2. Skewness is the unbiased sample coefficient; kurtosis is unbiased Fisher excess kurtosis. Constant-series shape coefficients and insufficient-sample values remain undefined. We retain tied modes and explicitly mark all-unique series. These conventions differ deliberately from Phase 3's ddof=0 scaling fit. Counts describe source observations, not distinct people across years.

In [3]:
statistics = pd.read_csv(REPORTS / "numeric_summary.csv")
display(statistics[statistics.source.eq("five_feature_profiles")])
print("Global measure summaries:", len(statistics))
print("Stratified measure summaries:", len(pd.read_csv(REPORTS / "stratified_numeric_summary.csv")))

,source,scope,measure,n,missing,zeros,nonzero,distinct_values,mean,median,...,q1,q3,iqr,skewness,excess_kurtosis,mode_values,mode_frequency,mode_count,mode_status,shape_status
126,five_feature_profiles,selected_state_year,occupancy_ratio,437,0,0,437,433,1.189940,1.103260,...,0.808238,1.423792,0.615553,1.566316,4.981243,"[0.275, 0.55, 0.7152103559870551, 1.0]",2,4,multiple_modes,defined
127,five_feature_profiles,selected_state_year,undertrial_share,437,0,0,437,421,0.664764,0.656228,...,0.587604,0.751006,0.163402,-0.285606,1.720328,[1.0],14,1,single_mode,defined
128,five_feature_profiles,selected_state_year,female_share,437,0,20,417,409,0.038224,0.036292,...,0.026396,0.048511,0.022115,0.623203,1.138909,[0.0],20,1,single_mode,defined
129,five_feature_profiles,selected_state_year,age_18_30_share,437,0,3,434,433,0.466552,0.454276,...,0.400798,0.515337,0.114540,0.438525,3.671952,[0.0],3,1,single_mode,defined
130,five_feature_profiles,selected_state_year,illiterate_share,437,0,10,427,426,0.276484,0.291490,...,0.191167,0.365441,0.174274,-0.113165,-0.135553,[0.0],10,1,single_mode,defined


Global measure summaries: 131
Stratified measure summaries: 2659


## Year-wise trends with coverage control

All-available totals contain 34 jurisdictions through 2008 and 35 thereafter. The balanced series retains the same 34 throughout 2001–2013. Pooled occupancy is total prisoners / total capacity, not the unweighted mean of state ratios. Neither series counts unique individuals over time.

In [4]:
annual = pd.read_csv(REPORTS / "year_summary_available.csv")
balanced = pd.read_csv(REPORTS / "year_summary_balanced.csv")
display(annual)
display(balanced.iloc[[0, -1]])
first, last = balanced.iloc[0], balanced.iloc[-1]
print(f"Fixed-panel prisoner-count change: {100 * (last.prisoners / first.prisoners - 1):.2f}%")
print(f"Fixed-panel capacity change: {100 * (last.capacity / first.capacity - 1):.2f}%")
print(f"Fixed-panel pooled occupancy: {100 * first.occupancy_ratio_pooled:.2f}% to {100 * last.occupancy_ratio_pooled:.2f}%")

,cohort,year,states,prisoners,capacity,convicts,undertrials,detenues,others,female_prisoners,occupancy_ratio_pooled,occupancy_ratio_mean_state,undertrial_share_pooled,female_share_pooled,states_above_capacity,zero_prisoner_states,prisoners_change,prisoners_change_fraction
0,all_available_states,2001,34,313635,229713,75663,220817,3510,13645,11094,1.365334,1.297368,0.704057,0.035372,22,0,NaN,NaN
1,all_available_states,2002,34,322357,229874,82121,223038,4832,12366,12047,1.402320,1.329681,0.691897,0.037372,21,0,8722.0,0.027809
2,all_available_states,2003,34,326519,233543,91766,217658,4008,13087,12780,1.398111,1.327755,0.666601,0.039140,23,0,4162.0,0.012911
3,all_available_states,2004,34,331391,235012,98527,217130,4491,11243,12970,1.410102,1.228917,0.655208,0.039138,20,1,4872.0,0.014921
4,all_available_states,2005,34,358368,246497,108572,237076,2542,10178,13986,1.453843,1.225712,0.661543,0.039027,22,0,26977.0,0.081405
5,all_available_states,2006,34,373271,263911,116675,245244,2275,9077,14657,1.414382,1.194720,0.657013,0.039266,24,1,14903.0,0.041586
6,all_available_states,2007,34,376396,277304,120115,250727,4687,867,15401,1.357341,1.191125,0.666126,0.040917,24,0,3125.0,0.008372
7,all_available_states,2008,34,384753,297777,123307,257928,2978,540,15929,1.292084,1.137214,0.670373,0.041401,21,0,8357.0,0.022203
8,all_available_states,2009,35,376969,307052,123941,250204,2232,592,15406,1.227704,1.128564,0.663726,0.040868,20,0,-7784.0,-0.020231
9,all_available_states,2010,35,368998,320450,125789,240098,2325,786,15037,1.151499,1.088525,0.650676,0.040751,17,0,-7971.0,-0.021145


,cohort,year,states,prisoners,capacity,convicts,undertrials,detenues,others,female_prisoners,occupancy_ratio_pooled,occupancy_ratio_mean_state,undertrial_share_pooled,female_share_pooled,states_above_capacity,zero_prisoner_states,prisoners_change,prisoners_change_fraction
0,same_states_all_13_years,2001,34,313635,229713,75663,220817,3510,13645,11094,1.365334,1.297368,0.704057,0.035372,22,0,NaN,NaN
12,same_states_all_13_years,2013,34,411900,347803,129597,278422,3113,768,18186,1.184291,1.025644,0.675946,0.044151,18,1,26836.0,0.069692


Fixed-panel prisoner-count change: 31.33%
Fixed-panel capacity change: 51.41%
Fixed-panel pooled occupancy: 136.53% to 118.43%


Faster capacity growth than inmate-count growth accounts arithmetically for lower pooled occupancy in this fixed panel. It does not establish a policy's causal effect. Source definitions and errors still limit interpretation.

In [5]:
latest = pd.read_csv(REPORTS / "latest_state_snapshot.csv")
latest_year = int(latest.year.max())
display(latest[["state", "total_prisoners", "total_capacity", "occupancy_ratio", "undertrial_share"]].head(10))
display(latest.sort_values("occupancy_ratio", ascending=False)[["state", "total_prisoners", "total_capacity", "occupancy_ratio"]].head(5))
print("Snapshot year:", latest_year)
print("Highest-count state's fraction of available prisoners:", latest.iloc[0].total_prisoners / latest.total_prisoners.sum())

,state,total_prisoners,total_capacity,occupancy_ratio,undertrial_share
0,Uttar Pradesh,83518,48550,1.720247,0.695658
1,Madhya Pradesh,34708,26967,1.287055,0.506166
2,Bihar,31259,35681,0.876068,0.851243
3,Punjab,27449,18629,1.473455,0.674196
4,Maharashtra,27400,24544,1.116362,0.705511
5,West Bengal,22778,20585,1.106534,0.723110
6,Rajasthan,19293,16622,1.160691,0.686570
7,Jharkhand,18220,14243,1.279225,0.771021
8,Haryana,17644,16964,1.040085,0.602074
9,Chhattisgarh,15840,6070,2.609555,0.583396


,state,total_prisoners,total_capacity,occupancy_ratio
9,Chhattisgarh,15840,6070,2.609555
14,Delhi,13552,6250,2.168320
0,Uttar Pradesh,83518,48550,1.720247
31,Arunachal Pradesh,92,56,1.642857
24,Meghalaya,850,530,1.603774


Snapshot year: 2013
Highest-count state's fraction of available prisoners: 0.20271752849569893


## Prisoner categories and demographics

Each marginal distribution uses its own covered prisoner population. Gender, status, age, nationality, education, caste, religion and domicile are separate dimensions and must not be added together. Their distributions do not establish individual risk or disparity relative to the general population without suitable external denominators.

In [6]:
demographic_year = pd.read_csv(REPORTS / "demographic_year.csv")
current = demographic_year[demographic_year.year.eq(latest_year)]
display(current[current.dimension.isin(["gender", "legal_status"])])
display(current[~current.dimension.isin(["gender", "legal_status"])])

,source,dimension,year,category,prisoners,covered_prisoners,covered_states,share_of_covered_population,population_coverage_fraction
271,Jail wise population of prison inmates.csv,gender,2013,female_prisoners,18188,411992,35,0.044146,1.0
272,Jail wise population of prison inmates.csv,gender,2013,male_prisoners,393804,411992,35,0.955854,1.0
321,Jail wise population of prison inmates.csv,legal_status,2013,convicts,129608,411992,35,0.314589,1.0
322,Jail wise population of prison inmates.csv,legal_status,2013,detenues,3113,411992,35,0.007556,1.0
323,Jail wise population of prison inmates.csv,legal_status,2013,others,768,411992,35,0.001864,1.0
324,Jail wise population of prison inmates.csv,legal_status,2013,under_trial,278503,411992,35,0.675991,1.0


,source,dimension,year,category,prisoners,covered_prisoners,covered_states,share_of_covered_population,population_coverage_fraction
48,Age_group.csv,age_group,2013,age_16_18,33,389214,34,0.000085,0.944713
49,Age_group.csv,age_group,2013,age_18_30,163478,389214,34,0.420021,0.944713
50,Age_group.csv,age_group,2013,age_30_50,174717,389214,34,0.448897,0.944713
51,Age_group.csv,age_group,2013,age_50_above,50986,389214,34,0.130997,0.944713
76,Age_group.csv,nationality,2013,Foreigners,3871,389214,34,0.009946,0.944713
77,Age_group.csv,nationality,2013,Indians,385343,389214,34,0.990054,0.944713
126,Caste.csv,caste,2013,OBC,130242,411992,35,0.316127,1.000000
127,Caste.csv,caste,2013,Others,144955,411992,35,0.351839,1.000000
128,Caste.csv,caste,2013,SC,89489,411992,35,0.217211,1.000000
129,Caste.csv,caste,2013,ST,47306,411992,35,0.114823,1.000000


The 2013 age distribution excludes West Bengal because of its incomplete age record. Other 2013 marginals may cover all 35 available jurisdictions. Read covered_states, covered_prisoners and population_coverage_fraction with each result. Coverage=1 means the full available reference, not independently certified completeness of all Indian prisons.

In [7]:
status_distribution = pd.read_csv(REPORTS / "demographic_status_year.csv")
gender_distribution = pd.read_csv(REPORTS / "demographic_gender_year.csv")
display(status_distribution[status_distribution.source.eq("Education.csv") & status_distribution.year.eq(latest_year) & status_distribution.legal_status.eq("under_trial")])
display(gender_distribution[gender_distribution.source.eq("Education.csv") & gender_distribution.year.eq(latest_year)])
print("These conditional distributions use dimensions jointly observed in the same source.")

,year,legal_status,category,prisoners,status_population,share_within_status,covered_states,source,dimension
618,2013,under_trial,Below Class X,117373,278503,0.421442,35,Education.csv,education
619,2013,under_trial,Class X and above but below graduate,56806,278503,0.203969,35,Education.csv,education
620,2013,under_trial,Graduate,16233,278503,0.058287,35,Education.csv,education
621,2013,under_trial,Holding technical degree/diploma etc,2642,278503,0.009486,35,Education.csv,education
622,2013,under_trial,Illiterate,80393,278503,0.288661,35,Education.csv,education
623,2013,under_trial,Post-Graduate,5056,278503,0.018154,35,Education.csv,education


,source,dimension,year,gender,category,prisoners,gender_population,covered_states,share_within_gender
482,Education.csv,education,2013,Female,Below Class X,6788,18188,35,0.373213
483,Education.csv,education,2013,Female,Class X and above but below graduate,2166,18188,35,0.119090
484,Education.csv,education,2013,Female,Graduate,645,18188,35,0.035463
485,Education.csv,education,2013,Female,Holding technical degree/diploma etc,102,18188,35,0.005608
486,Education.csv,education,2013,Female,Illiterate,8251,18188,35,0.453651
487,Education.csv,education,2013,Female,Post-Graduate,236,18188,35,0.012976
488,Education.csv,education,2013,Male,Below Class X,167857,393804,35,0.426245
489,Education.csv,education,2013,Male,Class X and above but below graduate,82964,393804,35,0.210673
490,Education.csv,education,2013,Male,Graduate,21740,393804,35,0.055205
491,Education.csv,education,2013,Male,Holding technical degree/diploma etc,4148,393804,35,0.010533


These conditional distributions use dimensions jointly observed in the same source.


## Correlation analysis

Pearson and Spearman coefficients describe aggregate association. Separate-year and within-state-centered results provide complementary views. They do not remove all confounding or make repeated years independent. Shared denominators and differences in state size can contribute to correlations; no p-values or causal claims are made.

In [8]:
correlations = pd.read_csv(REPORTS / "correlations.csv")
pooled = correlations[correlations.scope.eq("selected_profiles_pooled") & correlations.method.eq("pearson")]
display(pooled.pivot(index="left", columns="right", values="correlation"))
pair = correlations[correlations.left.eq("illiterate_share") & correlations.right.eq("occupancy_ratio")]
display(pair[pair.scope.isin(["selected_profiles_pooled", "selected_profiles_within_state_centered"]) | (pair.scope.eq("selected_profiles_within_year") & pair.year.eq(latest_year))])

right,age_18_30_share,female_share,illiterate_share,occupancy_ratio,undertrial_share
left,,,,,
age_18_30_share,1.000000,0.050887,0.070211,0.061511,0.235745
female_share,0.050887,1.000000,0.126308,-0.134527,-0.186921
illiterate_share,0.070211,0.126308,1.000000,0.354320,0.068009
occupancy_ratio,0.061511,-0.134527,0.354320,1.000000,0.004181
undertrial_share,0.235745,-0.186921,0.068009,0.004181,1.000000


,scope,year,method,left,right,paired_n,correlation
20,selected_profiles_pooled,NaN,pearson,illiterate_share,occupancy_ratio,437,0.354320
45,selected_profiles_pooled,NaN,spearman,illiterate_share,occupancy_ratio,437,0.414655
670,selected_profiles_within_year,2013.0,pearson,illiterate_share,occupancy_ratio,33,0.300547
695,selected_profiles_within_year,2013.0,spearman,illiterate_share,occupancy_ratio,33,0.249352
720,selected_profiles_within_state_centered,NaN,pearson,illiterate_share,occupancy_ratio,437,0.306423
745,selected_profiles_within_state_centered,NaN,spearman,illiterate_share,occupancy_ratio,437,0.255034


## Record frequencies and supporting indicators

Native record frequencies describe table structure; prisoner proportions use weighted counts. Releases, escape/rearrest, training and rehabilitation measures remain separate. Wages are not summed into national totals and their currency/time basis is unverified. Goods values retain the explicit source rupee units without inflation adjustment.

In [9]:
frequencies = pd.read_csv(REPORTS / "categorical_record_frequencies.csv")
display(frequencies[frequencies.source.eq("Education.csv") & frequencies.field.eq("education")])
coverage = pd.read_csv(REPORTS / "source_year_coverage.csv")
display(coverage[coverage.year.eq(latest_year)])
crime = pd.read_csv(REPORTS / "crime_year.csv")
display(crime[crime.year.eq(latest_year)].sort_values(["source", "reported_count"], ascending=[True, False]).groupby("source").head(3))

,source,field,category,native_records,denominator_records,record_fraction
248,Education.csv,education,Below Class X,888,5328,0.166667
249,Education.csv,education,Class X and above but below graduate,888,5328,0.166667
250,Education.csv,education,Graduate,888,5328,0.166667
251,Education.csv,education,Holding technical degree/diploma etc,888,5328,0.166667
252,Education.csv,education,Illiterate,888,5328,0.166667
253,Education.csv,education,Post-Graduate,888,5328,0.166667


,source,year,native_records,covered_states,reference_states,matched_reference_prisoners,population_coverage_fraction
12,Age_group.csv,2013,544,34,35,389214,0.944713
25,Caste.csv,2013,280,35,35,411992,1.000000
38,Death_sentence.csv,2013,35,35,35,411992,1.000000
51,Domicile.csv,2013,210,35,35,411992,1.000000
64,Education.csv,2013,420,35,35,411992,1.000000
77,Education_facilities.csv,2013,35,35,35,411992,1.000000
90,IPC_crime_inmates_convicted.csv,2013,735,35,35,411992,1.000000
103,IPC_crime_inmates_under_trial.csv,2013,735,35,35,411992,1.000000
116,Inmates_death.csv,2013,280,35,35,411992,1.000000
129,Inmates_escapee.csv,2013,140,35,35,411992,1.000000


,source,year,crime_head,reported_count,covered_states,within_source_total,share_within_source_counts
266,IPC_crime_inmates_convicted.csv,2013,MURDER,68199,35,113678,0.599931
269,IPC_crime_inmates_convicted.csv,2013,RAPE,8188,35,113678,0.072028
253,IPC_crime_inmates_convicted.csv,2013,ATTEMPT TO COMMIT MURDER,7010,35,113678,0.061665
539,IPC_crime_inmates_under_trial.csv,2013,MURDER,60110,35,226542,0.265337
545,IPC_crime_inmates_under_trial.csv,2013,THEFTS,25285,35,226542,0.111613
526,IPC_crime_inmates_under_trial.csv,2013,ATTEMPT TO COMMIT MURDER,23688,35,226542,0.104563
798,Other Prison data/SLL_Crime_headwise_distribut...,2013,N.D.P.S. ACT,6948,35,15822,0.439135
803,Other Prison data/SLL_Crime_headwise_distribut...,2013,REGISTRATION OF FOREIGNERS ACT,2060,35,15822,0.130198
787,Other Prison data/SLL_Crime_headwise_distribut...,2013,ARMS ACT,1722,35,15822,0.108836
1058,Other Prison data/SLL_crime_head_wise_distribu...,2013,N.D.P.S. ACT,16403,35,51895,0.316081


In [10]:
recidivism = pd.read_csv(REPORTS / "recidivism_state_year.csv")
display(recidivism[~recidivism.count_pair_valid])
display(pd.read_csv(REPORTS / "recidivism_year.csv"))
assert recidivism.loc[~recidivism.count_pair_valid, "habitual_share_of_admissions"].isna().all()
print("Habitual/admissions fractions remain undefined for zero admissions or contradictory counts.")
print("They are not longitudinal reoffending probabilities.")

,state,year,source_record,convicts_admitted,habitual_offenders,count_pair_valid,habitual_share_of_admissions
329,Andaman & Nicobar Islands,2002,331,0,19,False,NaN


,year,reported_states,reported_admissions,reported_habitual_offenders,valid_pair_states,invalid_pair_states,positive_denominator_states,ratio_cohort_admissions,ratio_cohort_habitual,habitual_share_of_admissions_pooled
0,2001,34,196064,9264,34,0,32,196064,9264,0.047250
1,2002,34,220909,12264,33,1,32,220909,12245,0.055430
2,2003,34,228693,16798,34,0,32,228693,16798,0.073452
3,2004,34,205521,14361,34,0,32,205521,14361,0.069876
4,2005,34,203610,10653,34,0,31,203610,10653,0.052321
5,2006,34,254901,13084,34,0,32,254901,13084,0.051330
6,2007,34,298815,13904,34,0,31,298815,13904,0.046530
7,2008,34,242632,11801,34,0,32,242632,11801,0.048637
8,2009,35,212785,11118,35,0,34,212785,11118,0.052250
9,2010,35,241992,10537,35,0,33,241992,10537,0.043543


Habitual/admissions fractions remain undefined for zero admissions or contradictory counts.
They are not longitudinal reoffending probabilities.


In [11]:
# reconcile counts and check all output/input hashes
assert annual[["convicts", "undertrials", "detenues", "others"]].sum(axis=1).eq(annual.prisoners).all()
np.testing.assert_allclose(annual.occupancy_ratio_pooled, annual.prisoners / annual.capacity)
for _, group in demographic_year.groupby(["source", "dimension", "year"]):
    assert group.prisoners.sum() == group.covered_prisoners.iloc[0]
    assert np.isclose(group.share_of_covered_population.sum(), 1)
manifest = json.loads((REPORTS / "table_manifest.json").read_text())
inputs = json.loads((REPORTS / "input_provenance.json").read_text())
assert all(sha256(REPORTS / r["file"]) == r["sha256"] for r in manifest)
assert all(sha256(ROOT / r["file"]) == r["sha256"] for r in inputs)
print("Verified", len(manifest), "tables and", len(inputs), "unchanged upstream files.")

Verified 22 tables and 85 unchanged upstream files.


## Findings and phase boundary

Location, dispersion, shape, frequency, state/year, demographic, prisoner-category and correlation summaries are now reproducible. The [descriptive report](../report/descriptive_analytics.md) records actual findings and links their supporting tables. Source errors, changing coverage, historical geography, sparse values, repeated state-years and ecological interpretation remain limitations.

Phase 4 ends here. Phase 5 requires approval before generating visualizations. This notebook runs no plots, mining algorithms, predictive models or Spark benchmarks.